In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

MODEL_READY_DIR = PROJECT_ROOT / "data" / "processed" / "model_ready"

required_files = [
    "mat_X_train_balanced.csv",
    "mat_y_train_balanced.csv",
    "mat_X_test_processed.csv",
    "mat_y_test.csv",
    "por_X_train_balanced.csv",
    "por_y_train_balanced.csv",
    "por_X_test_processed.csv",
    "por_y_test.csv",
]

for filename in required_files:
    path = MODEL_READY_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

mat_X_train_balanced.csv: FOUND
mat_y_train_balanced.csv: FOUND
mat_X_test_processed.csv: FOUND
mat_y_test.csv: FOUND
por_X_train_balanced.csv: FOUND
por_y_train_balanced.csv: FOUND
por_X_test_processed.csv: FOUND
por_y_test.csv: FOUND


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load datasets
mat_X_train = pd.read_csv(MODEL_READY_DIR / "mat_X_train_balanced.csv")
mat_y_train = pd.read_csv(MODEL_READY_DIR / "mat_y_train_balanced.csv")

mat_X_test = pd.read_csv(MODEL_READY_DIR / "mat_X_test_processed.csv")
mat_y_test = pd.read_csv(MODEL_READY_DIR / "mat_y_test.csv")

por_X_train = pd.read_csv(MODEL_READY_DIR / "por_X_train_balanced.csv")
por_y_train = pd.read_csv(MODEL_READY_DIR / "por_y_train_balanced.csv")

por_X_test = pd.read_csv(MODEL_READY_DIR / "por_X_test_processed.csv")
por_y_test = pd.read_csv(MODEL_READY_DIR / "por_y_test.csv")

# Flatten target columns into Series
mat_y_train = mat_y_train.iloc[:, 0]
mat_y_test = mat_y_test.iloc[:, 0]

por_y_train = por_y_train.iloc[:, 0]
por_y_test = por_y_test.iloc[:, 0]

# Display dataset dimensions
datasets = {
    "Mathematics train": (mat_X_train, mat_y_train),
    "Mathematics test": (mat_X_test, mat_y_test),
    "Portuguese train": (por_X_train, por_y_train),
    "Portuguese test": (por_X_test, por_y_test),
}

for name, (X, y) in datasets.items():
    print(f"\n{name}")
    print("Features:", X.shape)
    print("Target:", y.shape)
    print("Target distribution:")
    print(y.value_counts().sort_index())

    assert len(X) == len(y), f"Row mismatch in {name}"
    assert not X.isnull().any().any(), f"Missing values in {name}"
    assert np.isfinite(X.select_dtypes(include=np.number)).all().all(), (
        f"Non-finite numeric values in {name}"
    )

print("\nBasic dataset validation passed.")


Mathematics train
Features: (424, 56)
Target: (424,)
Target distribution:
performance
0    212
1    212
Name: count, dtype: int64

Mathematics test
Features: (79, 56)
Target: (79,)
Target distribution:
performance
0    26
1    53
Name: count, dtype: int64

Portuguese train
Features: (878, 56)
Target: (878,)
Target distribution:
performance
0    439
1    439
Name: count, dtype: int64

Portuguese test
Features: (130, 56)
Target: (130,)
Target distribution:
performance
0     20
1    110
Name: count, dtype: int64

Basic dataset validation passed.


In [3]:
# Verify that train and test feature columns match

for name, X_train, X_test in [
    ("Mathematics", mat_X_train, mat_X_test),
    ("Portuguese", por_X_train, por_X_test),
]:
    print(f"\n{name} dataset")

    print("Training feature count:", X_train.shape[1])
    print("Test feature count:", X_test.shape[1])

    train_cols = list(X_train.columns)
    test_cols = list(X_test.columns)

    missing_in_test = sorted(set(train_cols) - set(test_cols))
    extra_in_test = sorted(set(test_cols) - set(train_cols))

    print("Missing in test:", missing_in_test)
    print("Extra in test:", extra_in_test)

    assert train_cols == test_cols, (
        f"{name}: training and test feature columns differ"
    )

    assert X_train.select_dtypes(exclude="number").empty, (
        f"{name}: training data contains non-numeric features"
    )

    assert X_test.select_dtypes(exclude="number").empty, (
        f"{name}: test data contains non-numeric features"
    )

    print("Feature consistency verified.")

print("\nAll feature checks passed.")


Mathematics dataset
Training feature count: 56
Test feature count: 56
Missing in test: []
Extra in test: []
Feature consistency verified.

Portuguese dataset
Training feature count: 56
Test feature count: 56
Missing in test: []
Extra in test: []
Feature consistency verified.

All feature checks passed.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import joblib

print("All model-training imports completed successfully.")

All model-training imports completed successfully.


In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": SVC(
        probability=True,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ),

    "ANN": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=1000,
        early_stopping=True,
        random_state=42
    )
}

print("Models defined:", len(models))

for model_name in models:
    print("-", model_name)

Models defined: 6
- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- KNN
- ANN


In [6]:

# Check data types of training features

print("MATHEMATICS TRAINING FEATURES")
print(mat_X_train.dtypes.value_counts())
print("\nNon-numeric columns:")
print(mat_X_train.select_dtypes(exclude="number").columns.tolist())

print("\nPORTUGUESE TRAINING FEATURES")
print(por_X_train.dtypes.value_counts())
print("\nNon-numeric columns:")
print(por_X_train.select_dtypes(exclude="number").columns.tolist())


MATHEMATICS TRAINING FEATURES
float64    56
Name: count, dtype: int64

Non-numeric columns:
[]

PORTUGUESE TRAINING FEATURES
float64    56
Name: count, dtype: int64

Non-numeric columns:
[]


In [7]:

# Check target labels and missing values

targets = {
    "Mathematics training target": mat_y_train,
    "Mathematics test target": mat_y_test,
    "Portuguese training target": por_y_train,
    "Portuguese test target": por_y_test,
}

for name, target in targets.items():
    values = set(target.dropna().unique())

    print(f"\n{name}")
    print("Unique labels:", sorted(values))
    print("Missing labels:", target.isna().sum())
    print("Label counts:")
    print(target.value_counts(dropna=False).sort_index())

    assert target.notna().all(), f"{name} contains missing labels!"
    assert values == {0, 1}, f"{name} does not contain both binary labels!"

print("\nAll target checks passed.")



Mathematics training target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0    212
1    212
Name: count, dtype: int64

Mathematics test target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0    26
1    53
Name: count, dtype: int64

Portuguese training target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0    439
1    439
Name: count, dtype: int64

Portuguese test target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0     20
1    110
Name: count, dtype: int64

All target checks passed.


In [8]:

# Compare training and test sizes

print("MATHEMATICS")
print("Training features:", mat_X_train.shape)
print("Training target:", mat_y_train.shape)
print("Test features:", mat_X_test.shape)
print("Test target:", mat_y_test.shape)

print("\nPORTUGUESE")
print("Training features:", por_X_train.shape)
print("Training target:", por_y_train.shape)
print("Test features:", por_X_test.shape)
print("Test target:", por_y_test.shape)

# Basic consistency checks
assert len(mat_X_train) == len(mat_y_train)
assert len(mat_X_test) == len(mat_y_test)
assert len(por_X_train) == len(por_y_train)
assert len(por_X_test) == len(por_y_test)

print("\nFeature/target row counts match.")


MATHEMATICS
Training features: (424, 56)
Training target: (424,)
Test features: (79, 56)
Test target: (79,)

PORTUGUESE
Training features: (878, 56)
Training target: (878,)
Test features: (130, 56)
Test target: (130,)

Feature/target row counts match.


In [9]:

print("MATHEMATICS NON-NUMERIC COLUMNS:")
print(mat_X_train.select_dtypes(exclude="number").columns.tolist())

print("\nPORTUGUESE NON-NUMERIC COLUMNS:")
print(por_X_train.select_dtypes(exclude="number").columns.tolist())


MATHEMATICS NON-NUMERIC COLUMNS:
[]

PORTUGUESE NON-NUMERIC COLUMNS:
[]


In [10]:

for name, target in {
    "Math train": mat_y_train,
    "Math test": mat_y_test,
    "Portuguese train": por_y_train,
    "Portuguese test": por_y_test,
}.items():
    print(f"\n{name}")
    print("Labels:", sorted(target.dropna().unique()))
    print("Missing values:", target.isna().sum())
    print(target.value_counts(dropna=False).sort_index())



Math train
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0    212
1    212
Name: count, dtype: int64

Math test
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0    26
1    53
Name: count, dtype: int64

Portuguese train
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0    439
1    439
Name: count, dtype: int64

Portuguese test
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0     20
1    110
Name: count, dtype: int64


In [11]:

# Load the prepared CSV files for model training

mat_X_train = pd.read_csv(MODEL_READY_DIR / "mat_X_train_balanced.csv")
mat_y_train = pd.read_csv(MODEL_READY_DIR / "mat_y_train_balanced.csv").squeeze("columns")

mat_X_test = pd.read_csv(MODEL_READY_DIR / "mat_X_test_processed.csv")
mat_y_test = pd.read_csv(MODEL_READY_DIR / "mat_y_test.csv").squeeze("columns")

por_X_train = pd.read_csv(MODEL_READY_DIR / "por_X_train_balanced.csv")
por_y_train = pd.read_csv(MODEL_READY_DIR / "por_y_train_balanced.csv").squeeze("columns")

por_X_test = pd.read_csv(MODEL_READY_DIR / "por_X_test_processed.csv")
por_y_test = pd.read_csv(MODEL_READY_DIR / "por_y_test.csv").squeeze("columns")

print("All eight CSV files loaded successfully.")

print("\nMATHEMATICS")
print("Training features:", mat_X_train.shape)
print("Training target:", mat_y_train.shape)
print("Test features:", mat_X_test.shape)
print("Test target:", mat_y_test.shape)

print("\nPORTUGUESE")
print("Training features:", por_X_train.shape)
print("Training target:", por_y_train.shape)
print("Test features:", por_X_test.shape)
print("Test target:", por_y_test.shape)


All eight CSV files loaded successfully.

MATHEMATICS
Training features: (424, 56)
Training target: (424,)
Test features: (79, 56)
Test target: (79,)

PORTUGUESE
Training features: (878, 56)
Training target: (878,)
Test features: (130, 56)
Test target: (130,)


In [1]:

# Cell: Import libraries and define baseline models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Reproducibility
RANDOM_STATE = 42

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "SVM": SVC(
        probability=True,
        random_state=RANDOM_STATE
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "ANN": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=1000,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
}

print(f"Models defined: {len(models)}")
for name, model in models.items():
    print(f"- {name}: {type(model).__name__}")


Models defined: 6
- Logistic Regression: LogisticRegression
- Decision Tree: DecisionTreeClassifier
- Random Forest: RandomForestClassifier
- SVM: SVC
- KNN: KNeighborsClassifier
- ANN: MLPClassifier


In [2]:

# Cell: Train and evaluate all six models on both datasets

from pathlib import Path
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Store results in the project directory
PROJECT_ROOT = MODEL_READY_DIR.parents[2]
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Prepare both datasets
datasets = {
    "Mathematics": (
        mat_X_train,
        mat_y_train,
        mat_X_test,
        mat_y_test
    ),
    "Portuguese": (
        por_X_train,
        por_y_train,
        por_X_test,
        por_y_test
    )
}

results = []
trained_models = {}

for dataset_name, (X_train, y_train, X_test, y_test) in datasets.items():

    print(f"\n{'=' * 60}")
    print(f"DATASET: {dataset_name}")
    print(f"{'=' * 60}")

    # Ensure feature columns match
    assert list(X_train.columns) == list(X_test.columns), (
        f"Feature mismatch in {dataset_name}"
    )

    # Ensure there are no missing values
    assert not X_train.isnull().values.any(), (
        f"Missing values in {dataset_name} training features"
    )
    assert not X_test.isnull().values.any(), (
        f"Missing values in {dataset_name} test features"
    )

    for model_name, model in models.items():

        print(f"Training {model_name}...")

        # Train only on the training data
        model.fit(X_train, y_train)

        # Predict test labels
        y_pred = model.predict(X_test)

        # Calculate ROC-AUC using probabilities when available
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_test)
        else:
            y_score = None

        # Compute evaluation metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, zero_division=0
        )

        # ROC-AUC requires both classes in the test set
        if y_score is not None and len(set(y_test)) == 2:
            roc_auc = roc_auc_score(y_test, y_score)
        else:
            roc_auc = float("nan")

        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1_Score": f1,
            "ROC_AUC": roc_auc
        })

        # Keep fitted models separate by dataset and model
        trained_models[(dataset_name, model_name)] = model

        print(
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f} | "
            f"ROC-AUC: {roc_auc:.4f}"
        )

# Create a results table
results_df = pd.DataFrame(results)

# Sort within each dataset by F1-score
results_df = results_df.sort_values(
    by=["Dataset", "F1_Score"],
    ascending=[True, False]
).reset_index(drop=True)

# Save baseline results
results_file = RESULTS_DIR / "baseline_model_comparison.csv"
results_df.to_csv(results_file, index=False)

print("\nBASELINE MODEL COMPARISON")
display(results_df.round(4))

print(f"\nResults saved to: {results_file}")
print(f"Total model evaluations: {len(results_df)}")


NameError: name 'MODEL_READY_DIR' is not defined

In [3]:

# ============================================================
# PHASE 12: BASELINE MODEL TRAINING AND EVALUATION
# Student Performance Prediction Using ML
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Find project root and prepared dataset directory
# ------------------------------------------------------------

cwd = Path.cwd().resolve()

# Supports running Jupyter from the project root or notebooks/
if (cwd / "data" / "processed" / "model_ready").is_dir():
    PROJECT_ROOT = cwd
elif (cwd / "model_ready").is_dir() and cwd.parent.name == "processed":
    PROJECT_ROOT = cwd.parents[2]
elif (cwd / "data" / "processed").is_dir() and cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "processed" / "model_ready").is_dir() and cwd.name == "data":
    PROJECT_ROOT = cwd.parent
else:
    # Search current directory and its parents
    PROJECT_ROOT = None
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "processed" / "model_ready").is_dir():
            PROJECT_ROOT = candidate
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/model_ready. "
        "Check that your prepared CSV files are in the project folder."
    )

MODEL_READY_DIR = PROJECT_ROOT / "data" / "processed" / "model_ready"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Prepared data directory:", MODEL_READY_DIR)

# ------------------------------------------------------------
# 2. Load all eight prepared CSV files
# ------------------------------------------------------------

required_files = [
    "mat_X_train_balanced.csv",
    "mat_y_train_balanced.csv",
    "mat_X_test_processed.csv",
    "mat_y_test.csv",
    "por_X_train_balanced.csv",
    "por_y_train_balanced.csv",
    "por_X_test_processed.csv",
    "por_y_test.csv"
]

missing_files = [
    filename for filename in required_files
    if not (MODEL_READY_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing prepared files: {missing_files}\n"
        f"Expected location: {MODEL_READY_DIR}"
    )

mat_X_train = pd.read_csv(
    MODEL_READY_DIR / "mat_X_train_balanced.csv"
)
mat_y_train = pd.read_csv(
    MODEL_READY_DIR / "mat_y_train_balanced.csv"
).squeeze("columns")

mat_X_test = pd.read_csv(
    MODEL_READY_DIR / "mat_X_test_processed.csv"
)
mat_y_test = pd.read_csv(
    MODEL_READY_DIR / "mat_y_test.csv"
).squeeze("columns")

por_X_train = pd.read_csv(
    MODEL_READY_DIR / "por_X_train_balanced.csv"
)
por_y_train = pd.read_csv(
    MODEL_READY_DIR / "por_y_train_balanced.csv"
).squeeze("columns")

por_X_test = pd.read_csv(
    MODEL_READY_DIR / "por_X_test_processed.csv"
)
por_y_test = pd.read_csv(
    MODEL_READY_DIR / "por_y_test.csv"
).squeeze("columns")

print("\nAll eight CSV files loaded successfully.")

# ------------------------------------------------------------
# 3. Validate the data
# ------------------------------------------------------------

datasets = {
    "Mathematics": (
        mat_X_train, mat_y_train, mat_X_test, mat_y_test
    ),
    "Portuguese": (
        por_X_train, por_y_train, por_X_test, por_y_test
    )
}

for dataset_name, (X_train, y_train, X_test, y_test) in datasets.items():
    print(f"\n{dataset_name}")
    print("Training features:", X_train.shape)
    print("Training target:", y_train.shape)
    print("Test features:", X_test.shape)
    print("Test target:", y_test.shape)

    assert len(X_train) == len(y_train), (
        f"{dataset_name}: training feature/target row mismatch"
    )
    assert len(X_test) == len(y_test), (
        f"{dataset_name}: test feature/target row mismatch"
    )
    assert list(X_train.columns) == list(X_test.columns), (
        f"{dataset_name}: training and test features differ"
    )
    assert not X_train.isna().any().any(), (
        f"{dataset_name}: missing training feature values"
    )
    assert not X_test.isna().any().any(), (
        f"{dataset_name}: missing test feature values"
    )
    assert set(pd.Series(y_train).unique()).issubset({0, 1}), (
        f"{dataset_name}: unexpected training labels"
    )
    assert set(pd.Series(y_test).unique()).issubset({0, 1}), (
        f"{dataset_name}: unexpected test labels"
    )

print("\nData validation passed.")

# ------------------------------------------------------------
# 4. Define the six baseline models
# ------------------------------------------------------------

RANDOM_STATE = 42

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "SVM": SVC(
        probability=True,
        random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    "ANN": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=1000,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
}

print(f"\nModels defined: {len(models)}")
for model_name in models:
    print("-", model_name)

# ------------------------------------------------------------
# 5. Train and evaluate
# ------------------------------------------------------------

results = []
trained_models = {}

for dataset_name, (X_train, y_train, X_test, y_test) in datasets.items():

    print(f"\n{'=' * 65}")
    print(f"DATASET: {dataset_name.upper()}")
    print(f"{'=' * 65}")

    for model_name, model_template in models.items():

        print(f"Training {model_name}...", end=" ")

        # Use a fresh estimator for each dataset
        from sklearn.base import clone
        model = clone(model_template)

        # Fit using training data only
        model.fit(X_train, y_train)

        # Predict test labels
        y_pred = model.predict(X_test)

        # Obtain scores for ROC-AUC
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_test)
        else:
            y_score = None

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, zero_division=0
        )

        if y_score is not None and len(np.unique(y_test)) == 2:
            roc_auc = roc_auc_score(y_test, y_score)
        else:
            roc_auc = np.nan

        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1_Score": f1,
            "ROC_AUC": roc_auc
        })

        trained_models[(dataset_name, model_name)] = model

        print(
            f"Accuracy={accuracy:.4f}, "
            f"Precision={precision:.4f}, "
            f"Recall={recall:.4f}, "
            f"F1={f1:.4f}, "
            f"ROC-AUC={roc_auc:.4f}"
        )

# ------------------------------------------------------------
# 6. Display and save comparison results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["Dataset", "F1_Score"],
    ascending=[True, False]
).reset_index(drop=True)

results_file = RESULTS_DIR / "baseline_model_comparison.csv"
results_df.to_csv(results_file, index=False)

print("\n" + "=" * 65)
print("BASELINE MODEL COMPARISON")
print("=" * 65)

display(results_df.round(4))

print("\nTotal model evaluations:", len(results_df))
print("Results saved to:", results_file)

# Confirm the expected number of evaluations
assert len(results_df) == 12, (
    "Expected 12 evaluations: 6 models x 2 datasets."
)
print("Evaluation completed successfully.")


Project root: E:\PROJECTS\Mini Project\Project\Student-Performance-Prediction
Prepared data directory: E:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\model_ready

All eight CSV files loaded successfully.

Mathematics
Training features: (424, 56)
Training target: (424,)
Test features: (79, 56)
Test target: (79,)

Portuguese
Training features: (878, 56)
Training target: (878,)
Test features: (130, 56)
Test target: (130,)

Data validation passed.

Models defined: 6
- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- KNN
- ANN

DATASET: MATHEMATICS
Training Logistic Regression... Accuracy=0.6203, Precision=0.7347, Recall=0.6792, F1=0.7059, ROC-AUC=0.6118
Training Decision Tree... Accuracy=0.6709, Precision=0.7755, Recall=0.7170, F1=0.7451, ROC-AUC=0.6470
Training Random Forest... Accuracy=0.6709, Precision=0.7143, Recall=0.8491, F1=0.7759, ROC-AUC=0.6589
Training SVM... Accuracy=0.6709, Precision=0.7455, Recall=0.7736, F1=0.7593, ROC-AUC=0.6190
Tra

,Dataset,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,Mathematics,Random Forest,0.6709,0.7143,0.8491,0.7759,0.6589
1,Mathematics,SVM,0.6709,0.7455,0.7736,0.7593,0.6190
2,Mathematics,ANN,0.6456,0.7193,0.7736,0.7455,0.6836
3,Mathematics,Decision Tree,0.6709,0.7755,0.7170,0.7451,0.6470
4,Mathematics,Logistic Regression,0.6203,0.7347,0.6792,0.7059,0.6118
5,Mathematics,KNN,0.5443,0.7576,0.4717,0.5814,0.5922
6,Portuguese,Random Forest,0.8000,0.8621,0.9091,0.8850,0.6757
7,Portuguese,SVM,0.7846,0.8796,0.8636,0.8716,0.6814
8,Portuguese,ANN,0.7615,0.8835,0.8273,0.8545,0.6027
9,Portuguese,Decision Tree,0.7462,0.8598,0.8364,0.8479,0.5432



Total model evaluations: 12
Results saved to: E:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\results\tables\baseline_model_comparison.csv
Evaluation completed successfully.


Phase 13: Hyperparameter Tuning

In [4]:

from pathlib import Path

SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"

required_split_files = [
    "mat_X_train.csv",
    "mat_X_test.csv",
    "mat_y_train.csv",
    "mat_y_test.csv",
    "por_X_train.csv",
    "por_X_test.csv",
    "por_y_train.csv",
    "por_y_test.csv"
]

for filename in required_split_files:
    path = SPLIT_DIR / filename
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {filename}")


FOUND: mat_X_train.csv
FOUND: mat_X_test.csv
FOUND: mat_y_train.csv
FOUND: mat_y_test.csv
FOUND: por_X_train.csv
FOUND: por_X_test.csv
FOUND: por_y_train.csv
FOUND: por_y_test.csv


In [6]:

# ============================================================
# PHASE 13: HYPERPARAMETER TUNING WITH CATEGORICAL ENCODING
# GridSearchCV + StratifiedKFold + SMOTE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / "data" / "processed" / "splits").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/splits."
    )

SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Load original split files
# ------------------------------------------------------------

datasets = {
    "Mathematics": {
        "X_train": pd.read_csv(SPLIT_DIR / "mat_X_train.csv"),
        "X_test": pd.read_csv(SPLIT_DIR / "mat_X_test.csv"),
        "y_train": pd.read_csv(
            SPLIT_DIR / "mat_y_train.csv"
        ).squeeze("columns"),
        "y_test": pd.read_csv(
            SPLIT_DIR / "mat_y_test.csv"
        ).squeeze("columns")
    },
    "Portuguese": {
        "X_train": pd.read_csv(SPLIT_DIR / "por_X_train.csv"),
        "X_test": pd.read_csv(SPLIT_DIR / "por_X_test.csv"),
        "y_train": pd.read_csv(
            SPLIT_DIR / "por_y_train.csv"
        ).squeeze("columns"),
        "y_test": pd.read_csv(
            SPLIT_DIR / "por_y_test.csv"
        ).squeeze("columns")
    }
}

for name, data in datasets.items():
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_train = data["y_train"]
    y_test = data["y_test"]

    assert list(X_train.columns) == list(X_test.columns), (
        f"{name}: train/test columns do not match"
    )
    assert len(X_train) == len(y_train)
    assert len(X_test) == len(y_test)
    assert not X_train.isna().any().any(), (
        f"{name}: missing training features"
    )
    assert not X_test.isna().any().any(), (
        f"{name}: missing test features"
    )
    assert set(y_train.unique()).issubset({0, 1}), (
        f"{name}: unexpected training labels"
    )
    assert set(y_test.unique()).issubset({0, 1}), (
        f"{name}: unexpected test labels"
    )

    print(f"\n{name}")
    print("Training shape:", X_train.shape)
    print("Test shape:", X_test.shape)
    print("Training labels:", y_train.value_counts().to_dict())
    print("Categorical columns:",
          X_train.select_dtypes(
              include=["object", "category", "bool"]
          ).columns.tolist())

# ------------------------------------------------------------
# 3. Define model grids
# ------------------------------------------------------------

model_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=42),
        {
            "model__C": [0.1, 1.0, 10.0]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10],
            "model__min_samples_leaf": [1, 2]
        }
    ),

    "SVM": (
        SVC(probability=True, random_state=42),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__kernel": ["rbf", "linear"]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ["uniform", "distance"]
        }
    ),

    "ANN": (
        MLPClassifier(
            max_iter=500,
            early_stopping=True,
            random_state=42
        ),
        {
            "model__hidden_layer_sizes": [(32,), (64, 32)],
            "model__alpha": [0.0001, 0.01]
        }
    )
}

# ------------------------------------------------------------
# 4. Tune each model using preprocessing inside CV
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

tuning_results = []
test_results = []
best_estimators = {}

for dataset_name, data in datasets.items():

    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    # Detect feature types from training data only
    categorical_cols = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_cols = [
        col for col in X_train.columns
        if col not in categorical_cols
    ]

    # Encode categories and scale numeric columns
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_cols
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_cols
            )
        ],
        remainder="drop"
    )

    print("\n" + "=" * 65)
    print("DATASET:", dataset_name)
    print("=" * 65)

    for model_name, (estimator, param_grid) in model_grids.items():

        print(f"\nTuning {model_name}...")

        # All transformations and SMOTE occur within each CV fold.
        # Note: SMOTE after one-hot encoding can create fractional
        # one-hot values; see methodological note below.
        pipeline = Pipeline([
            ("preprocess", preprocessor),
            ("smote", SMOTE(random_state=42, k_neighbors=5)),
            ("model", estimator)
        ])

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
            error_score="raise"
        )

        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        best_estimators[(dataset_name, model_name)] = best_model

        tuning_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Best_CV_F1": search.best_score_,
            "Best_Params": str(search.best_params_)
        })

        # Evaluate on the original untouched test set
        y_pred = best_model.predict(X_test)
        y_score = best_model.predict_proba(X_test)[:, 1]

        test_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(
                y_test, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_test, y_pred, zero_division=0
            ),
            "F1_Score": f1_score(
                y_test, y_pred, zero_division=0
            ),
            "ROC_AUC": (
                roc_auc_score(y_test, y_score)
                if len(np.unique(y_test)) == 2
                else np.nan
            )
        })

        print(f"Best CV F1: {search.best_score_:.4f}")
        print("Best parameters:", search.best_params_)

# ------------------------------------------------------------
# 5. Save and display results
# ------------------------------------------------------------

tuning_df = pd.DataFrame(tuning_results)
test_results_df = pd.DataFrame(test_results)

tuning_file = RESULTS_DIR / "hyperparameter_tuning_results.csv"
test_file = RESULTS_DIR / "tuned_model_test_results.csv"

tuning_df.to_csv(tuning_file, index=False)
test_results_df.to_csv(test_file, index=False)

print("\nHYPERPARAMETER TUNING RESULTS")
display(tuning_df.round(4))

print("\nTUNED MODEL TEST RESULTS")
display(
    test_results_df.sort_values(
        ["Dataset", "F1_Score"],
        ascending=[True, False]
    ).round(4)
)

print("\nSaved:", tuning_file)
print("Saved:", test_file)
print("Completed searches:", len(tuning_df))
print("Expected searches: 12")



Mathematics
Training shape: (316, 30)
Test shape: (79, 30)
Training labels: {1: 212, 0: 104}
Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Portuguese
Training shape: (519, 30)
Test shape: (130, 30)
Training labels: {1: 439, 0: 80}
Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

DATASET: Mathematics

Tuning Logistic Regression...
Best CV F1: 0.7485
Best parameters: {'model__C': 10.0}

Tuning Decision Tree...
Best CV F1: 0.7343
Best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 4}

Tuning Random Forest...
Best CV F1: 0.7895
Best parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}

Tuning SVM...
Best CV F1: 0.7907
Best paramet

,Dataset,Model,Best_CV_F1,Best_Params
0,Mathematics,Logistic Regression,0.7485,{'model__C': 10.0}
1,Mathematics,Decision Tree,0.7343,"{'model__max_depth': 5, 'model__min_samples_le..."
2,Mathematics,Random Forest,0.7895,"{'model__max_depth': 10, 'model__min_samples_l..."
3,Mathematics,SVM,0.7907,"{'model__C': 0.1, 'model__kernel': 'rbf'}"
4,Mathematics,KNN,0.5568,"{'model__n_neighbors': 5, 'model__weights': 'u..."
5,Mathematics,ANN,0.6993,"{'model__alpha': 0.0001, 'model__hidden_layer_..."
6,Portuguese,Logistic Regression,0.8809,{'model__C': 10.0}
7,Portuguese,Decision Tree,0.8946,"{'model__max_depth': 5, 'model__min_samples_le..."
8,Portuguese,Random Forest,0.9287,"{'model__max_depth': 10, 'model__min_samples_l..."
9,Portuguese,SVM,0.9263,"{'model__C': 10.0, 'model__kernel': 'rbf'}"



TUNED MODEL TEST RESULTS


,Dataset,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
3,Mathematics,SVM,0.7215,0.7460,0.8868,0.8103,0.7199
1,Mathematics,Decision Tree,0.7342,0.8077,0.7925,0.8000,0.5929
5,Mathematics,ANN,0.6962,0.7636,0.7925,0.7778,0.6858
2,Mathematics,Random Forest,0.6709,0.7143,0.8491,0.7759,0.6466
0,Mathematics,Logistic Regression,0.5949,0.7234,0.6415,0.6800,0.6060
4,Mathematics,KNN,0.5443,0.7576,0.4717,0.5814,0.5922
9,Portuguese,SVM,0.8154,0.8839,0.9000,0.8919,0.6655
8,Portuguese,Random Forest,0.7846,0.8534,0.9000,0.8761,0.6841
7,Portuguese,Decision Tree,0.7538,0.8750,0.8273,0.8505,0.5934
11,Portuguese,ANN,0.7308,0.8713,0.8000,0.8341,0.6091



Saved: E:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\results\tables\hyperparameter_tuning_results.csv
Saved: E:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\results\tables\tuned_model_test_results.csv
Completed searches: 12
Expected searches: 12
